# Behavior Coordinator — Explosive Breaching Demo

VLM-driven behavior coordinator using Qwen2.5-VL served via vLLM.

| Config | Role |
|---|---|
| `config.json` | Plan the full behavior sequence from the task description |
| `config_goto.json` | Extract GOTO navigation parameters |
| `config_scan.json` | Extract SCAN target objects (+ optional scene image) |
| `config_receive_object.json` | Extract RECEIVE OBJECT parameters |

## 1. Imports

In [ ]:
import os
import re
import sys
import json
from pathlib import Path
from typing import List, Optional

import numpy as np
import rclpy
from rclpy.node import Node
from rclpy.qos import QoSProfile, QoSReliabilityPolicy, QoSHistoryPolicy
from geometry_msgs.msg import Point
from behavior_msgs.msg import (
    AI2RCommandMessage,
    AI2RStatusMessage,
    AI2RNavigationMessage,
    AI2RReceiveObjectMessage,
)

# Make sure vlm_interface.py is importable from this notebook's directory
CONFIG_DIR = Path(".").resolve()
sys.path.insert(0, str(CONFIG_DIR))
from vlm_interface import VLMInterface

print("Imports loaded.")

## 2. VLM Interfaces

Load one `VLMInterface` per config. This also auto-detects the model ID from the running vLLM server.

In [ ]:
vlm_planner = VLMInterface(CONFIG_DIR / "config.json")
vlm_goto    = VLMInterface(CONFIG_DIR / "config_goto.json")
vlm_scan    = VLMInterface(CONFIG_DIR / "config_scan.json")
vlm_receive = VLMInterface(CONFIG_DIR / "config_receive_object.json")

print(f"Model: {vlm_planner.model}")
print(f"\nLoaded configs:")
for name, vlm in [("planner", vlm_planner), ("goto", vlm_goto),
                  ("scan",    vlm_scan),    ("receive", vlm_receive)]:
    print(f"  {name:8s}  type={vlm.prompt_type}  vision={vlm.supports_vision}  image={vlm.image}")

## 3. Helper Functions

In [ ]:
def parse_scene(msg):
    """Return (object_names, available_behaviors) from a status message."""
    names     = [obj.object_name for obj in msg.objects] if msg.objects else []
    behaviors = list(msg.available_behaviors) if msg.available_behaviors else []
    return names, behaviors


def log_failure(msg, log_file="failure_info.json"):
    """Extract failure info, append to JSON log, and return the info dict."""
    if msg.failed_behavior == "-":
        return None

    failure = msg.failure
    pos_err = failure.position_error
    norm    = float(np.linalg.norm([pos_err.x, pos_err.y, pos_err.z]))

    info = {
        "failed_behavior": msg.failed_behavior,
        "action_name":     failure.action_name,
        "action_type":     failure.action_type,
        "missing_frame":   failure.reference_frame if failure.missing_frame else None,
        "collision_with":  failure.collision_name  if failure.collision_name != "-" else None,
        "position_error":  norm if norm > failure.position_tolerance else None,
    }

    data = []
    if os.path.exists(log_file):
        with open(log_file) as f:
            try:
                data = json.load(f)
            except json.JSONDecodeError:
                data = []
    data.append(info)
    with open(log_file, "w") as f:
        json.dump(data, f, indent=4)

    return info


def parse_json_response(response: str) -> dict:
    """
    Parse a JSON object from a VLM response that may be wrapped in markdown code fences.
    Handles both:
        ```json { ... } ```
        { ... }
    """
    stripped = re.sub(r"^```[a-z]*\n?", "", response.strip(), flags=re.IGNORECASE)
    stripped = re.sub(r"\n?```$", "", stripped.strip())
    stripped = stripped.replace("'", '"')
    return json.loads(stripped)


def behavior_list_to_planqueue(response: str) -> List[List[str]]:
    """
    Parse VLM output into [[behavior_name, full_step], ...].
    Expects: behavior_list = [ BEHAVIOR (description), ... ]
    """
    match = re.search(r"behavior_list\s*=\s*\[(.*?)\]", response, re.DOTALL)
    if not match:
        print("Warning: no behavior_list found in VLM response.")
        return []

    list_block = match.group(1)
    items = re.findall(r"([^,]+(?:\([^\)]*\))?)", list_block)
    steps = [s.strip().rstrip(",") for s in items if s.strip()]

    queue = []
    for step in steps:
        m = re.match(r"^([A-Z][A-Z ]*?)(?:\s*\(|$)", step)
        if m:
            queue.append([m.group(1).strip(), step])
    return queue


def _point_to_numpy(point: Point) -> np.ndarray:
    return np.array([point.x, point.y, point.z])


def _get_pose_by_name(name, scene_names, scene_poses, robot_pose=None):
    if name.lower() == "robot" and robot_pose:
        return robot_pose
    if name in ("-", ""):
        return None
    try:
        return scene_poses[scene_names.index(name)]
    except ValueError:
        return None


def select_target_object(base_name, spatially_related_object, spatial_relation,
                         class_discriminator, scene_object_names, scene_object_positions,
                         robot_pose=None):
    """Resolve an ambiguous base name to a specific scene object using spatial geometry."""
    candidates = [
        (name, pose)
        for name, pose in zip(scene_object_names, scene_object_positions)
        if name.startswith(base_name) and name[len(base_name):].isdigit()
    ]
    if not candidates:
        return None
    if len(candidates) == 1:
        return candidates[0][0]

    ref_name = spatially_related_object if spatially_related_object not in ("-", "") else "Robot"
    ref_pose = _get_pose_by_name(ref_name, scene_object_names, scene_object_positions, robot_pose)
    if not ref_pose:
        return None

    if spatial_relation in ("DEFAULT", "-", ""):
        key = lambda x: np.linalg.norm(_point_to_numpy(x[1]) - _point_to_numpy(ref_pose))
        return (min if class_discriminator == "CLOSE" else max)(candidates, key=key)[0]

    ref_pos   = _point_to_numpy(ref_pose)
    robot_pos = _point_to_numpy(robot_pose) if robot_pose else ref_pos
    direction = robot_pos - ref_pos
    if np.linalg.norm(direction) < 1e-6:
        return candidates[0][0]

    dir_norm = direction / np.linalg.norm(direction)
    left_vec = np.cross(np.array([0.0, 0.0, 1.0]), dir_norm[:3])
    if np.linalg.norm(left_vec) > 1e-6:
        left_vec /= np.linalg.norm(left_vec)

    qualified = []
    for name, pose in candidates:
        offset = _point_to_numpy(pose) - ref_pos
        if   spatial_relation == "BEHIND" and np.dot(offset, dir_norm) < -0.1:
            qualified.append((name, np.linalg.norm(offset)))
        elif spatial_relation == "FRONT"  and np.dot(offset, dir_norm) >  0.1:
            qualified.append((name, np.linalg.norm(offset)))
        elif spatial_relation == "RIGHT"  and np.dot(offset, left_vec) >  0.5:
            qualified.append((name, np.linalg.norm(offset)))
        elif spatial_relation == "LEFT"   and np.dot(offset, left_vec) < -0.5:
            qualified.append((name, np.linalg.norm(offset)))

    if not qualified:
        return None
    return (min if class_discriminator == "CLOSE" else max)(qualified, key=lambda x: x[1])[0]


print("Helper functions defined.")

## 4. Test: Mission Planner

Simulate what the coordinator sends to the VLM on the first status message.

In [ ]:
sample_scene      = ["Person1", "Barrier1", "Charge1", "DoorPanel1", "DoorPullHandle1"]
sample_behaviors  = ["GOTO", "SCAN", "RECEIVE OBJECT", "PLACE CHARGE ON DOOR"]

vlm_input = (
    f"scene_objects: {sample_scene}\n"
    f"available_behaviors: {sample_behaviors}\n"
    f"previously_executed: \n"
    f"failed_behaviors: "
)

vlm_planner.first_log_interaction(vlm_input)
response = vlm_planner.call_model(vlm_input)
print("VLM plan response:")
print(response)

plan_queue = behavior_list_to_planqueue(response)
print("\nParsed plan queue:")
for i, (behavior, description) in enumerate(plan_queue):
    print(f"  {i+1}. [{behavior}]  {description}")

## 5. Test: GOTO Parameters

Pass a GOTO step description to the VLM and inspect the extracted navigation parameters.

In [ ]:
goto_scene       = ["Person1", "Barrier1", "Charge1", "DoorPanel1"]
goto_description = "GOTO (to the person to the right of the barrier)"

vlm_input = (
    f"scene_objects: {goto_scene}\n"
    f"task_description: {goto_description}"
)

response = vlm_goto.call_model(vlm_input)
print("Raw response:")
print(response)

params = parse_json_response(response)
print("\nParsed GOTO params:")
for k, v in params.items():
    print(f"  {k}: {v}")

## 6. Test: SCAN Parameters (with image)

The scan config includes a default image path (`vlm/test/room_expo.png`).
Override it with `image_path=` to pass a live frame, or leave `None` to use the config default.

In [ ]:
scan_scene       = ["Person1", "Barrier1", "Charge1", "DoorPanel1"]
scan_description = "SCAN"

vlm_input = (
    f"scene_objects: {scan_scene}\n"
    f"task_description: {scan_description}"
)

# Uses the image from config_scan.json by default.
# To pass a live frame:  vlm_scan.call_model(vlm_input, image_path="/path/to/frame.png")
# To force text-only:    vlm_scan.call_model(vlm_input, image_path="")
response = vlm_scan.call_model(vlm_input)
print("Raw response:")
print(response)

params = parse_json_response(response)
print("\nParsed SCAN params:")
print("  target_objects:", params["target_objects"])

## 7. Test: RECEIVE OBJECT Parameters

In [ ]:
receive_scene       = ["Person1", "Charge1"]
receive_description = "RECEIVE OBJECT (the explosive charge from the person)"

vlm_input = (
    f"scene_objects: {receive_scene}\n"
    f"task_description: {receive_description}"
)

response = vlm_receive.call_model(vlm_input)
print("Raw response:")
print(response)

params = parse_json_response(response)
print("\nParsed RECEIVE OBJECT params:")
print(f"  object_name: {params['object_name']}")
print(f"  side:        {params['side']}  ({'right' if params['side'] == 1 else 'left'})")

## 8. Behavior Coordinator Node

ROS2 node that ties the VLM interfaces together into a reactive coordinator:
- Calls the **mission planner** once on the first status message
- Advances through the plan queue, calling a **param extractor** for each behavior
- Waits for the robot to become idle before sending the next command

In [ ]:
class BehaviorCoordinator(Node):
    def __init__(self):
        super().__init__("behavior_coordination_node")

        self.vlm_planner = VLMInterface(CONFIG_DIR / "config.json")
        self.vlm_goto    = VLMInterface(CONFIG_DIR / "config_goto.json")
        self.vlm_scan    = VLMInterface(CONFIG_DIR / "config_scan.json")
        self.vlm_receive = VLMInterface(CONFIG_DIR / "config_receive_object.json")

        self.plan_queue       = []
        self.initialized      = False
        self.logged_failure   = False
        self.next_behavior    = ""
        self.next_description = ""
        self.last_completion  = None

        qos_be = QoSProfile(
            reliability=QoSReliabilityPolicy.BEST_EFFORT,
            history=QoSHistoryPolicy.KEEP_LAST, depth=1)
        qos_rel = QoSProfile(
            reliability=QoSReliabilityPolicy.RELIABLE,
            history=QoSHistoryPolicy.KEEP_LAST, depth=1)

        self.status_sub = self.create_subscription(
            AI2RStatusMessage, "/ihmc/behavior_tree/ai2r_status", self.on_status, qos_be)
        self.command_pub = self.create_publisher(
            AI2RCommandMessage, "/ihmc/behavior_tree/ai2r_command", qos_rel)

        print(f"BehaviorCoordinator ready  (model: {self.vlm_planner.model})")

    # ------------------------------------------------------------------
    # Status callback
    # ------------------------------------------------------------------

    def on_status(self, msg):
        scene_names, available_behaviors = parse_scene(msg)

        if not self.initialized:
            print(f"Scene objects:       {scene_names}")
            print(f"Available behaviors: {available_behaviors}")

        if msg.completed_behavior != "-" and msg.completed_behavior != self.last_completion:
            print(f"Completed: {msg.completed_behavior}")
            self.last_completion = msg.completed_behavior

        if msg.failed_behavior != "-" and not self.logged_failure:
            info = log_failure(msg)
            print(f"[FAILURE] {json.dumps(info, indent=2)}")
            self.logged_failure = True

        if msg.behavior_in_progress != "-":
            return
        if self.next_behavior and msg.completed_behavior != self.next_behavior:
            return

        if not self.plan_queue:
            if self.initialized:
                print("Mission complete. All behaviors executed.")
                return
            self._plan_mission(msg)

        if not self.plan_queue:
            return

        self.next_behavior, self.next_description = self.plan_queue.pop(0)
        print(f"Commanding [{self.next_behavior}]: {self.next_description}  "
              f"({len(self.plan_queue)} remaining)")

        cmd = self._build_command(self.next_behavior, self.next_description, msg)
        self.command_pub.publish(cmd)
        self.initialized    = True
        self.logged_failure = False

    # ------------------------------------------------------------------
    # Mission planning
    # ------------------------------------------------------------------

    def _plan_mission(self, msg):
        scene_names, available_behaviors = parse_scene(msg)
        vlm_input = (
            f"scene_objects: {scene_names}\n"
            f"available_behaviors: {available_behaviors}\n"
            f"previously_executed: {msg.completed_behavior if msg.completed_behavior != '-' else ''}\n"
            f"failed_behaviors: {msg.failed_behavior if msg.failed_behavior != '-' else ''}"
        )
        print("Calling VLM mission planner...")
        self.vlm_planner.first_log_interaction(vlm_input)
        response = self.vlm_planner.call_model(vlm_input)
        print(f"VLM plan:\n{response}")
        self.plan_queue = behavior_list_to_planqueue(response)
        if not self.plan_queue:
            print("ERROR: VLM returned no parseable behavior list.")

    # ------------------------------------------------------------------
    # Command builders
    # ------------------------------------------------------------------

    def _build_command(self, behavior, description, msg):
        cmd = AI2RCommandMessage()
        cmd.behavior_to_execute = behavior
        cmd.adapting_behavior   = False
        if   behavior == "SCAN":           return self._build_scan(cmd, description, msg)
        elif behavior == "GOTO":           return self._build_goto(cmd, description, msg)
        elif behavior == "RECEIVE OBJECT": return self._build_receive_object(cmd, description, msg)
        return cmd

    def _build_scan(self, cmd, description, msg):
        # SCAN has no extra message parameters — the robot does a general scene scan.
        # We still call the VLM to log which objects it expects to find.
        scene_names, _ = parse_scene(msg)
        vlm_input = f"scene_objects: {scene_names}\ntask_description: {description}"
        response  = self.vlm_scan.call_model(vlm_input)   # uses image from config by default
        print(f"SCAN expected targets: {response}")
        return cmd

    def _build_goto(self, cmd, description, msg):
        cmd.adapting_behavior = True
        scene_names, _ = parse_scene(msg)
        scene_poses    = [obj.object_pose_in_world.position for obj in msg.objects]
        robot_pos      = msg.robot_mid_feet_under_pelvis_pose_in_world.position
        vlm_input = f"scene_objects: {scene_names}\ntask_description: {description}"
        response  = self.vlm_goto.call_model(vlm_input)
        print(f"GOTO params: {response}")
        try:
            data = parse_json_response(response)
        except json.JSONDecodeError as e:
            print(f"Could not parse GOTO response ({e}).")
            return cmd
        target = data["target_object"]
        if target not in scene_names:
            target = select_target_object(
                base_name                = data["target_object"],
                spatially_related_object = data["spatially_related_object"],
                spatial_relation         = data["spatial_relation_obj"],
                class_discriminator      = data["class_discriminator"],
                scene_object_names       = scene_names,
                scene_object_positions   = scene_poses,
                robot_pose               = robot_pos,
            ) or data["target_object"]
        print(f"GOTO resolved target: {target}")
        nav = AI2RNavigationMessage()
        nav.target_object      = target
        nav.distance_to_object = 1.0
        nav.pov_object         = ""
        relation_str = data["spatial_relation_goto"]
        nav.spatial_relation = (
            getattr(AI2RNavigationMessage, relation_str)
            if hasattr(AI2RNavigationMessage, relation_str)
            else AI2RNavigationMessage.DEFAULT
        )
        if nav.spatial_relation == AI2RNavigationMessage.DEFAULT or not nav.pov_object:
            nav.pov_object = "walkingFrame"
        cmd.navigation = nav
        return cmd

    def _build_receive_object(self, cmd, description, msg):
        cmd.adapting_behavior = True
        scene_names, _ = parse_scene(msg)
        vlm_input = f"scene_objects: {scene_names}\ntask_description: {description}"
        response  = self.vlm_receive.call_model(vlm_input)
        print(f"RECEIVE OBJECT params: {response}")
        try:
            data = parse_json_response(response)
            receive_msg = AI2RReceiveObjectMessage()
            receive_msg.object_name = data["object_name"]
            receive_msg.side        = bytes([int(data["side"])])
            cmd.receive_object      = receive_msg
        except (json.JSONDecodeError, KeyError) as e:
            print(f"Could not parse RECEIVE OBJECT response ({e}).")
        return cmd


print("BehaviorCoordinator defined.")

## 9. Run the Demo

Initialize ROS2, create the coordinator, and spin. Interrupt the kernel to stop.

In [ ]:
rclpy.init()
node = BehaviorCoordinator()

try:
    print("Running explosive breaching demo... (interrupt kernel to stop)")
    rclpy.spin(node)
except KeyboardInterrupt:
    print("\nStopped by user.")
finally:
    node.destroy_node()
    if rclpy.ok():
        rclpy.shutdown()
    print("Shutdown complete.")